# Project Stages

## Preliminary Actions  

### **File 1:** `1. makeup_api_loading.ipynb`  
- Code implementation for data loading and saving.  
- Connecting to the API via URL: [https://makeup-api.herokuapp.com/api/v1/products.json](https://makeup-api.herokuapp.com/api/v1/products.json).  
- Saving the retrieved data to the file `makeup_all_products.json`.  

### **File 2:** `2. makeup_research_dataset.ipynb`  
- Data import  

## Current Actions  

### **File 3:** `3. makeup_data_preparation.ipynb`  
- Data processing and preparation for further analysis.  
- Applying necessary transformations and validations.  

## Upcoming Actions  

### **File 4:** `.4. makeup_color_table.ipynb`  
- Selecting a color palette to be used for chart creation.  
- Creating a color table for data visualization.  

### **File 5:** `5. makeup_project_visualization.ipynb`  
- Creating visualizations to analyze the data.  
- Displaying key correlations and statistical characteristics.  



## Actions Completed in the File  
**File Name:** `3.makeup_data_preparation.ipynb`

### Brief Overview of the Code Implemented in the File:

### **Code 1:** Data Loading  
- Imported data from the JSON file `makeup_all_products.json`.  
- Displayed the list of columns to verify the structure of the dataset.  

### **Code 2:** Data Splitting  
- Loaded data from the file `makeup_all_products.json`.  
- Assessed data integrity—ensured all key columns were present.  
- Split the dataset into three separate parts:  
  - **`makeup_valid_prices.csv`:** Products with valid prices (>0), prepared for analytics.  
  - **`makeup_no_prices.csv`:** Products with undefined prices (price = 0, missing, or NaN).  
  - **`makeup_additional_info.csv`:** Supplementary information, including links, API paths, etc.  
- Saved the resulting files in `.csv` format for further use.  

### **Code 3:** Missing Data Assessment  
- Checked for missing values in key columns to evaluate the need for their imputation.  

### **Code 4:** Filling Missing Values  
- Imported data from the file `makeup_valid_prices.csv`.  
- Filled missing values in the `category` and `brand` columns.  
- Saved the updated data to the file `makeup_valid_prices_filled.csv`.  

### **Code 5:** Post-Fill Validation  
- Evaluated the presence of missing values in the columns after filling (`category` and `brand`).  
- Source: `makeup_valid_prices_filled.csv`.  

### **Code 6:** Price Statistical Analysis  
- Analyzed data distribution in the `price` column to define categories.  
- Created a new column `categorize_price` to classify products by price.  
- Source: `makeup_valid_prices_filled.csv`.  

### **Code 7:** Adding New Columns  
- Added additional columns for analytics:  
  - `categorize_price`  
  - `brand_popularity`  
  - `category_product_type`  
  - `brand_category_product_type`  
- Saved the updated data to the file `makeup_valid_prices_filled_columns.csv`.  

### **Code 8:** Updated Data Validation  
- Evaluated missing values after adding new columns.  
- Source: `makeup_valid_prices_filled_columns.csv`.  

### **Code 9:** Unique Values Analysis  
- Counted the number of unique values for each column in the dataset.  
- Source: `makeup_valid_prices_filled_columns.csv`.  

### **Code 10:** Numerical Data Statistical Analysis  
- Assessed key statistical indicators for numerical columns.  


In [1]:
## **Code 1: Data Loading**

# Import libraries and load data, display general information.
import requests  # Library for making HTTP requests.
import json      # Library for working with JSON files.
import pandas as pd  # Library for data analysis (Pandas).

# Load data from a previously saved JSON file.
file_path = "makeup_all_products.json"  # Specify the path to the data file.
df = pd.read_json(file_path)    # Load data into a DataFrame for analysis.

# Display the list of columns.
columns = df.columns.tolist()  # Retrieve the list of columns.
print("List of columns in the DataFrame:")
print(columns)


List of columns in the DataFrame:
['id', 'brand', 'name', 'price', 'price_sign', 'currency', 'image_link', 'product_link', 'website_link', 'description', 'rating', 'category', 'product_type', 'tag_list', 'created_at', 'updated_at', 'product_api_url', 'api_featured_image', 'product_colors']


In [2]:
### Code 2: Data Splitting  
# Split the data into three parts and save them into separate .csv files.
# All datasets retain the 'id' column, preserving the data structure.

import pandas as pd
import numpy as np

# Load the JSON file.
file_path = "makeup_all_products.json"
df = pd.read_json(file_path)

# List of all expected initial columns.
full_columns = ['id', 'brand', 'name', 'price', 'price_sign', 'currency', 'image_link', 'product_link', 
                'website_link', 'description', 'rating', 'category', 'product_type', 'tag_list', 
                'created_at', 'updated_at', 'product_api_url', 'api_featured_image', 'product_colors']

# Check for the presence of all expected columns.
missing = [col for col in full_columns if col not in df.columns]
if missing:
    print("Missing columns:", missing)
else:
    print("All columns are present.")

# 1. Products with valid prices (price > 0).
df_valid = df[df["price"].apply(lambda x: pd.notna(x) and x > 0)].copy()

# 2. Products without defined prices: price == 0, NaN, or empty.
df_no_price = df[df["price"].isna() | (df["price"] == 0)].copy()

# 3. Additional data (not intended for analysis, saved to a separate file).
additional_columns = ['id', 'price_sign', 'currency', 'image_link', 'product_link', 'website_link', 
                      'tag_list', 'created_at', 'updated_at', 'product_api_url', 'api_featured_image']
df_additional = df[additional_columns].copy()

# Columns to retain in analytical datasets.
analysis_columns = ['id', 'brand', 'name', 'price', 'description', 'rating', 'category', 'product_type', 'product_colors']
df_valid = df_valid[analysis_columns]
df_no_price = df_no_price[analysis_columns]

# Save the datasets into CSV files.
df_valid.to_csv("makeup_valid_prices.csv", index=False)
df_no_price.to_csv("makeup_no_prices.csv", index=False)
df_additional.to_csv("makeup_additional_info.csv", index=False)

print("Files saved: makeup_valid_prices.csv, makeup_no_prices.csv, makeup_additional_info.csv")


All columns are present.
Files saved: makeup_valid_prices.csv, makeup_no_prices.csv, makeup_additional_info.csv


In [3]:
### Code 3: Missing Data Assessment  
# Evaluate the dataset for missing values to determine the need for imputation.

import pandas as pd

# Load data from the CSV file.
file_path = "makeup_valid_prices.csv"
df = pd.read_csv(file_path)

# Function to calculate missing values for each column.
def calculate_missing_values(dataframe):
    missing_stats = pd.DataFrame({
        'NaN_count': dataframe.isna().sum(),  # Count of NaN values.
        'empty_string_count': (dataframe == '').sum(),  # Count of empty strings.
        'zero_count': (dataframe == 0).sum(),  # Count of zero values.
    })
    missing_stats['total_missing'] = missing_stats.sum(axis=1)  # Total number of missing values across categories.
    return missing_stats

# Calculate missing values.
missing_values = calculate_missing_values(df)

# Display the results.
print("Missing value statistics for each column:")
print(missing_values)


Missing value statistics for each column:
                NaN_count  empty_string_count  zero_count  total_missing
id                      0                   0           0              0
brand                  12                   0           0             12
name                    0                   0           0              0
price                   0                   0           0              0
description            23                   0           0             23
rating                537                   0           0            537
category              405                   0           0            405
product_type            0                   0           0              0
product_colors          0                   0           0              0


In the following code, missing data will be filled:

Data from the file `makeup_valid_prices.csv` has been loaded.

Filling missing categories and brands in the `category` and `brand` columns:

### Filling Missing Values in the `category` Column:

1. **Automatic Filling by Product Type:** For each product type, the most common category among products with existing categories was identified. This allowed automatic filling of missing categories for all products with the same type.

2. **Manual Filling:** For certain product types, categories were filled manually based on knowledge of the product types:
   - `mascara` — category filled as `liquid`
   - `bronzer` — category filled as `powder`
   - `nail_polish` — category filled as `gel`
   - `eyebrow` — category filled as `pencil`

3. **Result:** All missing categories were filled, reducing the number of missing values in the `category` column to zero.

### Filling Missing Values in the `brand` Column:

Missing values in the `brand` column were filled using the `type_brand_map` dictionary, which associates the most common brand with each product type (`product_type`). If a product type was not found in the dictionary, the value remained as `np.nan`.

For example, if the `brand` column had a missing value (NaN) or an empty string, the function used the corresponding brand from `type_brand_map` based on the value in the `product_type` column.

### Outcome:

This approach successfully filled missing values in the `category` and `brand` columns, ensuring data completeness for further analysis.

The filled data has been saved to the file `makeup_valid_prices_filled.csv`.

In [4]:
### Code 4: Filling Missing Values  
# Fill missing values in the `category` and `brand` columns.

import pandas as pd
import numpy as np

# Load the dataset.
file_path = "makeup_valid_prices.csv"  # Specify the correct file path.
df = pd.read_csv(file_path)

# Check missing values in the 'category' and 'brand' columns.
print(f"Initial missing values in 'category': {df['category'].isna().sum()}")
print(f"Initial missing values in 'brand': {df['brand'].isna().sum()}")

# Filling categories.
manual_category_fallback = {
    'mascara': 'liquid',  # Example for mascara.
    'bronzer': 'powder',  # Example for bronzer.
    'nail_polish': 'gel',  # Example for nail_polish.
    'eyebrow': 'pencil',  # Example for eyebrow.
}

# Check missing values in the 'category' column.
empty_or_nan_cat = df['category'].isna() | (df['category'].str.strip() == '')

# Create a mapping: product_type → most common category.
type_cat_map = (
    df[~empty_or_nan_cat]
    .groupby('product_type')['category']
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

# Combine automatically generated map with manual fallback categories.
type_cat_map.update({k: v for k, v in manual_category_fallback.items() if k not in type_cat_map})

# Function to fill missing values in the 'category' column.
def fill_category(row):
    if pd.isna(row['category']) or row['category'].strip() == '':
        return type_cat_map.get(row['product_type'], np.nan)
    return row['category']

# Fill missing categories.
df['category'] = df.apply(fill_category, axis=1)

# Filling brands.
empty_or_nan_brand = df['brand'].isna() | (df['brand'].str.strip() == '')

# Create a mapping: product_type → most common brand (for missing value filling).
type_brand_map = (
    df[~empty_or_nan_brand]
    .groupby('product_type')['brand']
    .agg(lambda x: x.value_counts().index[0])  # Most common brand for each product type.
    .to_dict()
)

# Function to fill missing values in the 'brand' column.
def fill_brand(row):
    if pd.isna(row['brand']) or row['brand'].strip() == '':
        # Use the most common brand for the corresponding product type.
        return type_brand_map.get(row['product_type'], np.nan)
    return row['brand']

# Fill missing values in 'brand'.
df['brand'] = df.apply(fill_brand, axis=1)

# Check the number of missing values after processing.
print(f"Remaining missing values in 'category' after processing: {df['category'].isna().sum()}")
print(f"Remaining missing values in 'brand' after processing: {df['brand'].isna().sum()}")

# Save the result to a new file "makeup_valid_prices_filled.csv".
df.to_csv("makeup_valid_prices_filled.csv", index=False)


Initial missing values in 'category': 405
Initial missing values in 'brand': 12
Remaining missing values in 'category' after processing: 0
Remaining missing values in 'brand' after processing: 0


In [5]:
### Code 5: Post-Fill Validation  
# Check missing values for each column after filling the `category` and `brand` columns.

import pandas as pd

# Load the data from the CSV file.
file_path = "makeup_valid_prices_filled.csv"
df = pd.read_csv(file_path)

# Function to calculate missing values for each column.
def calculate_missing_values(dataframe):
    missing_stats = pd.DataFrame({
        'NaN_count': dataframe.isna().sum(),  # Count of NaN values.
        'empty_string_count': (dataframe == '').sum(),  # Count of empty strings.
        'zero_count': (dataframe == 0).sum(),  # Count of zero values.
    })
    missing_stats['total_missing'] = missing_stats.sum(axis=1)  # Total count of missing values.
    return missing_stats

# Calculate missing values.
missing_values = calculate_missing_values(df)

# Display the results.
print("Missing value statistics for each column:")
print(missing_values)


Missing value statistics for each column:
                NaN_count  empty_string_count  zero_count  total_missing
id                      0                   0           0              0
brand                   0                   0           0              0
name                    0                   0           0              0
price                   0                   0           0              0
description            23                   0           0             23
rating                537                   0           0            537
category                0                   0           0              0
product_type            0                   0           0              0
product_colors          0                   0           0              0


In [6]:
### Code 6: Price Statistical Analysis  
# Analyze statistics in the `price` column to define categories in the `categorize_price` column.

import pandas as pd

# Load data from the CSV file.
file_path = "makeup_valid_prices_filled.csv"
df = pd.read_csv(file_path)

# Filter data where the price is greater than 0.
filtered_prices = df[df['price'] > 0]['price']

# Calculate basic statistical metrics.
average_price = filtered_prices.mean()
min_price = filtered_prices.min()
max_price = filtered_prices.max()

# Print basic statistics.
print("Basic price statistics:")
print(f"Minimum price: {min_price}")
print(f"Average price: {average_price}")
print(f"Maximum price: {max_price}")
print()

# Calculate percentiles.
percentiles = {
    "25%": filtered_prices.quantile(0.25),
    "75%": filtered_prices.quantile(0.75),
    "99%": filtered_prices.quantile(0.99)
}

# Print percentiles.
print("Price percentiles:")
for label, value in percentiles.items():
    print(f"{label}: {value}")


Basic price statistics:
Minimum price: 1.99
Average price: 17.261550741163056
Maximum price: 77.0

Price percentiles:
25%: 9.5
75%: 23.0
99%: 50.24000000000001



### The following code adds new columns: 
`categorize_price`, `brand_popularity`, `category_product_type`, `brand_category_product_type`

---

### Creating the `categorize_price` Column:
The categorization is based on key percentiles:

- **`low` (<10):** Represents products below the 25% percentile. For convenience, prices are rounded to 10. (25%: ~9.5, rounded to 10).
  
- **`mid` (10–25):** Represents the mid-range segment between ~25% and 75% percentiles, including the median (range: 9.5 to 23.0).  

- **`high` (25–50):** Covers products in the range from the 75th to almost the 99th percentile (99%: ~50.24).  

- **`premium` (>50):** Represents the top 1% of products with very high prices. These are not considered outliers but belong to a distinct category.

---


In [7]:
### Code 7: Adding New Columns  
# Adding new columns: categorize_price, brand_popularity, category_product_type, brand_category_product_type.
# categorize_price categories: low (<10), mid (10–25), high (25–50), premium (>50).

import pandas as pd

# Load data from the CSV file.
file_path = "makeup_valid_prices_filled.csv"
df = pd.read_csv(file_path)

# 1. Categorize prices based on fixed thresholds.
def categorize_price(price):
    if price < 10:
        return 'low'
    elif price < 25:
        return 'mid'
    elif price < 50:
        return 'high'
    else:
        return 'premium'

df['price_category'] = df['price'].apply(categorize_price)

# 2. Calculate the number of products with valid prices within each brand.
brand_popularity = df.groupby('brand')['price'].count()
df['brand_popularity'] = df['brand'].map(brand_popularity)

# 3. Combine category and product type into a single column.
df['category_product_type'] = df['category'] + "_" + df['product_type']

# 4. Combine brand, category, and product type into a single column.
df['brand_category_product_type'] = df['brand'] + "_" + df['category'] + "_" + df['product_type']

# Save the updated data to a new file.
df.to_csv("makeup_valid_prices_filled_columns.csv", index=False)


In [8]:
### **Code 8:** Updated Data Validation  
# Check missing values for each column after adding new columns.

import pandas as pd

# Load data from the CSV file.
file_path = "makeup_valid_prices_filled_columns.csv"
df = pd.read_csv(file_path)

# Function to calculate missing values for each column.
def calculate_missing_values(dataframe):
    missing_stats = pd.DataFrame({
        'NaN_count': dataframe.isna().sum(),  # Count of NaN values.
        'empty_string_count': (dataframe == '').sum(),  # Count of empty strings.
        'zero_count': (dataframe == 0).sum(),  # Count of zero values.
    })
    missing_stats['total_missing'] = missing_stats.sum(axis=1)  # Total count of missing values.
    return missing_stats

# Calculate missing values.
missing_values = calculate_missing_values(df)

# Display the results.
print("Missing value statistics for each column:")
print(missing_values)


Missing value statistics for each column:
                             NaN_count  empty_string_count  zero_count  \
id                                   0                   0           0   
brand                                0                   0           0   
name                                 0                   0           0   
price                                0                   0           0   
description                         23                   0           0   
rating                             537                   0           0   
category                             0                   0           0   
product_type                         0                   0           0   
product_colors                       0                   0           0   
price_category                       0                   0           0   
brand_popularity                     0                   0           0   
category_product_type                0                   0           0

In [9]:
### Code 9: Unique Values Analysis  

import pandas as pd

# Load data from the CSV file.
file_path = "makeup_valid_prices_filled_columns.csv"
df = pd.read_csv(file_path)

# Count the number of unique values for each column.
unique_counts = df.nunique()

# Display the result.
print("Number of unique values for each column:")
print(unique_counts)


Number of unique values for each column:
id                             877
brand                           44
name                           877
price                          156
description                    820
rating                          26
category                        14
product_type                    10
product_colors                 625
price_category                   4
brand_popularity                24
category_product_type           26
brand_category_product_type    251
dtype: int64


In [10]:
### Code 10: Statistical Analysis of Numerical Data  

import pandas as pd

# Load data from the CSV file.
file_path = "makeup_valid_prices_filled_columns.csv"
df = pd.read_csv(file_path)

# Convert the 'price' column to numeric type.
df['price'] = pd.to_numeric(df['price'], errors='coerce')  # 'errors="coerce"' converts invalid entries to NaN.

# Generate statistical data for all numerical columns.
statistics = df.describe()

# Display statistical data.
print("Statistical data for numerical columns:")
print(statistics)


Statistical data for numerical columns:
                id       price      rating  brand_popularity
count   877.000000  877.000000  340.000000        877.000000
mean    507.942987   17.261551    4.319118         69.342075
std     302.830003   10.684513    0.675849         56.367286
min       1.000000    1.990000    1.500000          1.000000
25%     249.000000    9.500000    4.000000         27.000000
50%     487.000000   14.790000    4.500000         54.000000
75%     791.000000   23.000000    5.000000         93.000000
max    1048.000000   77.000000    5.000000        172.000000
